# Strong Lensing: Time Delay Accuracy and Precision

_Phil Marshall & Lynne Jones_

In the first [Time Delay Challenge](http://timedelaychallenge.org) paper, [Liao et al (2015)](http://arxiv.org/pdf/1409.1254.pdf) derived the following simple model for how strongly gravitationally lensed quasar time delay accuracy (A), precision (P) and success rate (f) depend on the night-to-night cadence, season and campaign length. 

\begin{align}
|A|_{\rm model} &\approx 0.06\% \left(\frac{\rm cad} {\rm 3 days}  \right)^{0.0}
                          \left(\frac{\rm sea}  {\rm 4 months}\right)^{-1.0}
                          \left(\frac{\rm camp}{\rm 5 years} \right)^{-1.1} \notag \\
  P_{\rm model} &\approx 4.0\% \left(\frac{\rm cad} {\rm 3 days}  \right)^{ 0.7}
                         \left(\frac{\rm sea}  {\rm 4 months}\right)^{-0.3}
                         \left(\frac{\rm camp}{\rm 5 years} \right)^{-0.6} \notag \\
  f_{\rm model} &\approx 30\% \left(\frac{\rm cad} {\rm 3 days}  \right)^{-0.4}
                        \left(\frac{\rm sea}  {\rm 4 months}\right)^{ 0.8}
                        \left(\frac{\rm camp}{\rm 5 years} \right)^{-0.2} \notag
\end{align}

The first two of these metrics are candidate Figure of Merit proxies, while one can imagine combining all three somehow to provide an approximate dark energy parameter Figure of Merit. These three metrics are implemented in [`TdcMetric.py`](https://github.com/lsst/rubin_sim/blob/main/rubin_sim/maf/metrics/seasonMetrics.py) of the  [rubin_sim](http://github.com/lsst/rubin_sim) git repository. 

The Time Delay Challenge developed these heuristics based on in-depth analysis of time delays in a relatively 'constant' survey strategy. Newer survey strategies have a fully implemented form of 'rolling cadence' that are likely to impact these heuristics. 

This notebook provides a demo calculation of these metrics. 

**Notebook overview (added for clarity).**

This notebook computes, per Healpix sky pixel, the three Liao et al. (2015) Time Delay Challenge (TDC)
heuristics for strongly-lensed quasars -- Accuracy (A), Precision (P) and success rate (f) -- as a
function of the night-to-night cadence, season length and campaign length delivered by a given LSST
cadence/OpSim run. The overall flow is:

1. Run a single 'complex' MAF metric (`maf.TdcMetric`) over a Healpix map of the whole sky. It computes
   cadence/season/campaign diagnostics from the visits in each pixel and combines them, via the model
   equations above, into A, P and f for that pixel.
2. Use `TdcMetric`'s 'reduce' functions to split the single complex result per pixel into separate
   Healpix maps (accuracy, precision, rate, cadence, season, campaign), each viewable/plottable on its own.
3. Post-process the maps into single numbers (means, a high-accuracy sky fraction/area, an approximate
   combined distance-precision Figure of Merit) suitable for a summary table.
4. Compare this metric ('SL TDC') across many different OpSim runs/strategies using MAF's run-comparison
   helper plots.

Note: this notebook does not define any custom Python functions (no `def`); all the logic below is
inline script code, which is why the added documentation takes the form of inline comments and markdown
explanations rather than function docstrings.

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

import rubin_sim.maf as maf  # rubin_sim.maf = the Metric Analysis Framework (MAF) used to evaluate an LSST cadence/OpSim run
from rubin_sim.data import (
    get_baseline,
)  # path to the baseline OpSim database, i.e. the simulated LSST cadence/strategy to analyze

### Run the metric, as well as some associated season cadence and length metrics.

In [ ]:
opsdb = get_baseline()
runName = os.path.split(opsdb)[-1].replace(".db", "")
print(runName)

In [ ]:
data_dir = None

if data_dir is None:
    import tempfile
    import os

    data_dir_itself = tempfile.TemporaryDirectory(prefix="02_TDC_TimeAccuracy_maf_", dir=os.getcwd())
    data_dir = data_dir_itself.name

print(f"Using the {data_dir_itself.name} directory for output of this notebook")

Note that our metric (`TdcMetric`) is actually a "complex" metric, as it calculates A (accuracy), P (precision), and f (rate) in one go (thus re-using the cadence/season/campaign values which must also be calculated for each set of visits), and then has 'reduce' methods that separate each of these individual results into separate values. 

It does implement minimum limiting magnitude values before 'counting' a visit, as well as adding dust extinction to the m5 per visit values.

In [ ]:
nside_tdc = 64  # Healpix resolution for the sky map (higher = finer pixels)
tdc_plots = [
    maf.HealpixSkyMap(),
    maf.HealpixHistogram(),
]  # default plot types: a sky map and a histogram of pixel values
plotDict = {
    "xMin": 0.01,
    "colorMin": 0.01,
    "percentileClip": 70,
    "nTicks": 5,
}  # shared plotting bounds/appearance
# summary metrics: reduce the whole-sky map of the (complex) TDC metric down to single numbers
tdc_summary = [
    maf.MeanMetric(),  # sky-averaged value
    maf.MedianMetric(),  # sky-median value (robust to outlier pixels)
    maf.RmsMetric(),  # scatter of the value across the sky
    maf.AreaThresholdMetric(
        upper_threshold=0.04
    ),  # sky area (fraction) where the metric value is below 0.04 (i.e. the 'high accuracy' area)
]
# Ideally need a way to do better on calculating the summary metrics for the high accuracy area.
slicer = maf.HealpixSlicer(
    nside=nside_tdc, use_cache=False
)  # slices the visit database into one set of visits per Healpix pixel
tdcMetric = maf.TdcMetric(metric_name="TDC")
# TdcMetric: a 'complex' per-pixel metric (rubin_sim.maf.metrics.seasonMetrics.TdcMetric). For each pixel it first
# derives the cadence (median time between visits within a season), season length, and campaign length from the
# visit timestamps (applying a limiting-magnitude cut per visit, and adding dust extinction to m5 via the DustMap
# below), then plugs cadence/season/campaign into the Liao et al. (2015) formulas above to get Accuracy (A),
# Precision (P) and success rate (f) in one go. Its 'reduce' methods later split this single complex result per
# pixel into separate maps: accuracy, precision, rate, cadence, season, campaign.
dustmap = maf.DustMap(
    nside=nside_tdc, interp=False
)  # per-pixel Galactic dust extinction map, used by TdcMetric to correct m5 depths
bundle = maf.MetricBundle(
    tdcMetric,
    slicer,
    constraint=None,  # no SQL constraint: use all visits (the metric itself applies its own magnitude cuts)
    run_name=runName,
    plot_dict=plotDict,
    plot_funcs=tdc_plots,
    maps_list=[dustmap],
    summary_metrics=tdc_summary,
)
bundles = {"TDC": bundle}

In [ ]:
outDir = data_dir
# MetricBundleGroup: groups the bundle(s), connects to the OpSim sqlite DB, runs the SQL query + metric
# calculation for every Healpix pixel, and (via run_all) also automatically calls TdcMetric's 'reduce'
# methods, which is why new entries (TDC_accuracy, TDC_precision, ...) appear in `bundles` afterwards.
g = maf.MetricBundleGroup(bundles, opsdb, out_dir=outDir)
g.run_all()

Note that we now have more bundles in our bundle dictionary. These new bundles contain the results of the reduce functions - so, the metrics A, P, f separately as well as the "cadence", "season" and "campaign" diagnostics. 

In [ ]:
bundles.keys()

If creating plots like this, we'd like to set the `plotDict` for each of these separately, so that we can get each plot to look "just right", and then we'll make the plots. When running in the script for the science radar, we can get approximately right by using `{'percentileClip': 70}` in the plotDict.

In [ ]:
# set a dedicated, sensible plotting range/label for each of the 6 reduced TDC quantities
# (accuracy, precision, rate, cadence, season, campaign), each stored as its own bundle 'TDC_<key>'
minVal = 0.01
maxVal = {"accuracy": 0.04, "precision": 10.0, "rate": 40, "cadence": 14, "season": 8.0, "campaign": 11.0}
units = {
    "accuracy": "%",
    "precision": "%",
    "rate": "%",
    "cadence": "days",
    "season": "months",
    "campaign": "years",
}
for key in maxVal:
    plotDict = {"xMin": minVal, "xMax": maxVal[key], "colorMin": minVal, "colorMax": maxVal[key]}
    plotDict["xlabel"] = "TDC %s (%s)" % (key, units[key])
    print(key, plotDict)
    bundles["TDC_%s" % (key)].set_plot_dict(
        plotDict
    )  # bundles auto-created by TdcMetric's reduce functions after run_all()

In [ ]:
g.plot_all(closefigs=False)

In [ ]:
bundle.metric_values

Let's do some post-processing to turn the metric maps into single numbers for a table in the observing strategy white paper. (in the science Radar, we can't do this as easily, so just use the median values of each summary value; this will be fairly close to the "high accuracy" parts of the sky, as it's most of it). 

In [ ]:
def get_valid_tdc_arrays(bundle):
    """Extract flat, unmasked arrays of the six TdcMetric quantities from a run MetricBundle.

    `TdcMetric` stores, in each Healpix pixel, a dict with the accuracy/precision/rate of strongly
    lensed quasar time delays together with the cadence/season/campaign diagnostics it was derived
    from. Pixels where the metric could not be computed (e.g. too few visits/seasons) are masked.
    This helper drops the masked pixels and unpacks the dicts into six flat 1D numpy arrays.

    Parameters
    ----------
    bundle : rubin_sim.maf.metricBundles.MetricBundle
        A MetricBundle built with ``maf.TdcMetric`` as its metric, already run (i.e.
        ``bundle.metric_values`` is populated by ``MetricBundleGroup.run_all()``). Each valid
        pixel's value is a dict with keys ``"accuracy"``, ``"precision"``, ``"rate"``,
        ``"cadence (days)"``, ``"season (months)"``, ``"campaign"``.

    Returns
    -------
    f, A, P, c, s, y : numpy.ndarray
        1D arrays (one entry per valid/unmasked Healpix pixel) of the success rate f (%),
        accuracy A (%), precision P (%), cadence c (days), season s (months) and campaign
        length y (years), respectively, in the same pixel order.
    """
    x = bundle.metric_values  # masked array, one element per Healpix pixel
    index = np.where(x.mask == False)  # keep only pixels where the metric could actually be computed
    f = np.array([each["rate"] for each in x[index]])
    A = np.array([each["accuracy"] for each in x[index]])
    P = np.array([each["precision"] for each in x[index]])
    c = np.array([each["cadence (days)"] for each in x[index]])
    s = np.array([each["season (months)"] for each in x[index]])
    y = np.array([each["campaign"] for each in x[index]])
    return f, A, P, c, s, y

In [ ]:
f, A, P, c, s, y = get_valid_tdc_arrays(bundle)
print(
    np.mean(f), np.mean(A), np.mean(P), np.mean(c), np.mean(s), np.mean(y)
)  # sky-averaged f, A, P, cadence, season, campaign

We are only interested in lenses with high accuracy delays, i.e. the fraction of the survey area where the _A_ metric is below some threshold. We can turn this into a sky area if we know the average size of a `HEALPix` pixel.

In [ ]:
def healpix_pixel_area_sqdeg(nside):
    """Solid angle of one Healpix pixel, in square degrees, for a given Healpix resolution.

    Parameters
    ----------
    nside : int
        Healpix ``Nside`` resolution parameter (the map has ``12 * nside**2`` equal-area pixels).

    Returns
    -------
    float
        Area of a single Healpix pixel, in square degrees.
    """
    npix = 12 * nside**2
    area_per_pixel_sr = 4 * np.pi / float(npix)  # steradians
    return area_per_pixel_sr * (180.0 / np.pi) ** 2  # square degrees


def summarize_high_accuracy_region(A, c, s, y, accuracy_threshold=0.04, nside=64):
    """Characterize the sky region where the TDC accuracy A is below a given threshold.

    We are only interested in lenses with high-accuracy time delays, i.e. the fraction of the
    survey area where the systematic accuracy bias A is below `accuracy_threshold`. This selects
    those pixels and summarizes the resulting sky fraction/area and the typical (median)
    cadence/season/campaign reached there.

    Parameters
    ----------
    A : numpy.ndarray
        Per-pixel accuracy values (%), e.g. from ``get_valid_tdc_arrays``.
    c, s, y : numpy.ndarray
        Per-pixel cadence (days), season (months) and campaign (years) arrays, in the same
        order/length as `A`, e.g. from ``get_valid_tdc_arrays``.
    accuracy_threshold : float, optional
        Accuracy threshold (%) defining the 'high accuracy' pixels (default 0.04%, i.e. 5 times
        better than the 0.2% threshold set by Hojjati & Linder 2014).
    nside : int, optional
        Healpix ``Nside`` used to produce `A`/`c`/`s`/`y`, needed to convert a pixel count into a
        sky area (default 64).

    Returns
    -------
    dict
        ``"high_accuracy_index"`` : numpy index array selecting the high-accuracy pixels in
        `A`/`c`/`s`/`y` (and, correspondingly, in `P`/`f` arrays from the same pixel set).
        ``"fraction"`` : percentage of the survey area in the high-accuracy region.
        ``"area_sqdeg"`` : sky area of the high-accuracy region, in square degrees.
        ``"median_cadence"``, ``"median_season"``, ``"median_campaign"`` : median cadence
        (days), season (months) and campaign (years) within the high-accuracy region.
    """
    high_accuracy = np.where(A < accuracy_threshold)
    fraction = 100 * (1.0 * len(A[high_accuracy])) / (1.0 * len(A))
    area_sqdeg = len(A[high_accuracy]) * healpix_pixel_area_sqdeg(nside)
    return {
        "high_accuracy_index": high_accuracy,
        "fraction": fraction,
        "area_sqdeg": area_sqdeg,
        "median_cadence": np.median(c[high_accuracy]),
        "median_season": np.median(s[high_accuracy]),
        "median_campaign": np.median(y[high_accuracy]),
    }

In [ ]:
accuracy_threshold = 0.04  # 5 times better than threshold of 0.2% set by Hojjati & Linder (2014).
hi_acc = summarize_high_accuracy_region(A, c, s, y, accuracy_threshold=accuracy_threshold, nside=64)

print(
    "Fraction of total survey area providing high accuracy time delays = ",
    np.round(hi_acc["fraction"], 1),
    "%",
)
print(
    "Median night-to-night cadence in high accuracy regions = ", np.round(hi_acc["median_cadence"], 1), "days"
)
print("Median season length in high accuracy regions = ", np.round(hi_acc["median_season"], 1), "months")
print("Median campaign length in high accuracy regions = ", int(hi_acc["median_campaign"]), "years")
print("Area of sky providing high accuracy time delays = ", int(hi_acc["area_sqdeg"]), "sq deg")

In [ ]:
hi_acc

In [ ]:
high_accuracy = hi_acc["high_accuracy_index"]

In [ ]:
Nside = 64
Npix = 12 * Nside**2
Area_per_pixel = 4 * np.pi / float(Npix)  # steradians
Area_per_pixel *= (180.0 / np.pi) * (180.0 / np.pi)  # square degrees
high_accuracy_area = len(A[high_accuracy]) * Area_per_pixel
print("Area of sky providing high accuracy time delays = ", int(high_accuracy_area), "sq deg")

In [ ]:
def compute_distance_precision_fom(
    P,
    f,
    high_accuracy_index,
    high_accuracy_area_sqdeg,
    modeling_error_pct=4.0,
    ref_area_sqdeg=18000.0,
    ref_rate_pct=30.0,
    ref_n_lenses=400,
):
    """Estimate a combined time-delay-cosmography distance-precision Figure of Merit.

    Combines the per-lens statistical precision `P` with a fixed lens-modeling systematic error,
    in quadrature, to get the total precision per lens. It then scales the TDC1 reference yield
    (`ref_n_lenses` lenses over `ref_area_sqdeg` sq deg at a `ref_rate_pct` success rate) to this
    survey's high-accuracy area and success rate to estimate the expected number of usable lenses,
    and finally averages the per-lens precision over that sample via sqrt(N) scaling, following
    the approach of Coe & Moustakas (2009). This overall precision is a reasonable proxy for the
    cosmological (e.g. H0) distance-precision Figure of Merit; Treu & Marshall (2016) quote an
    LSST-era target of 0.4-0.7%.

    Parameters
    ----------
    P : numpy.ndarray
        Per-pixel precision values (%), e.g. from ``get_valid_tdc_arrays``.
    f : numpy.ndarray
        Per-pixel success rate values (%), e.g. from ``get_valid_tdc_arrays``, same order as `P`.
    high_accuracy_index : numpy index array
        Index selecting the high-accuracy pixels within `P` and `f`, as returned by
        ``summarize_high_accuracy_region``.
    high_accuracy_area_sqdeg : float
        Sky area (sq deg) of the high-accuracy region, as returned by
        ``summarize_high_accuracy_region`` (key ``"area_sqdeg"``).
    modeling_error_pct : float, optional
        Assumed lens-modeling systematic error (%), added in quadrature to the measurement
        precision (default 4.0).
    ref_area_sqdeg, ref_rate_pct, ref_n_lenses : float, optional
        TDC1 reference survey area (sq deg), success rate (%) and lens yield used to scale the
        expected number of lenses for this survey (defaults: 18000 sq deg, 30%, 400 lenses).

    Returns
    -------
    dict
        ``"precision_per_lens"`` : combined measurement + modeling precision per lens (%).
        ``"n_lenses"`` : estimated number of usable lenses in the high-accuracy sample.
        ``"distance_precision"`` : combined percentage distance precision (the FoM proxy);
        0 if no usable lenses are found.
    """
    precision_per_lens = np.array([np.mean(P[high_accuracy_index]), modeling_error_pct])
    precision_per_lens = np.sqrt(np.sum(precision_per_lens * precision_per_lens))

    fraction = np.mean(f[high_accuracy_index])
    n_lenses = int((high_accuracy_area_sqdeg / ref_area_sqdeg) * (fraction / ref_rate_pct) * ref_n_lenses)

    distance_precision = (precision_per_lens * (n_lenses > 0)) / (np.sqrt(n_lenses) + (n_lenses == 0))

    return {
        "precision_per_lens": precision_per_lens,
        "n_lenses": n_lenses,
        "distance_precision": distance_precision,
    }

In [ ]:
fom = compute_distance_precision_fom(P, f, hi_acc["high_accuracy_index"], hi_acc["area_sqdeg"])

print(
    "Mean precision per lens in high accuracy sample, including modeling error = ",
    np.round(fom["precision_per_lens"], 2),
    "%",
)
print("Number of lenses in high accuracy sample = ", fom["n_lenses"])
print(
    "Maximum combined percentage distance precision (as in Coe & Moustakas 2009) = ",
    np.round(fom["distance_precision"], 2),
    "%",
)

The above overall precision can be related to the cosmological parameter precision, and so is a reasonable proxy Figure of Merit. This quantity is plotted by [Treu & Marshall (2016)](http://arxiv.org/abs/1605.05333) in their recent review: their target for the LSST era is between 0.4 and 0.7%.

# Conclusions

We no have good diagnostic and Figure of Merit metrics for assessing lens time delay measurement, based on extrapolating the TDC1 single filter catalog-level simulations. We see that we are *analysis-limited* in the sense that *if* we can combine the *ugrizy* light curves with such fidelity that they appear as if we had simply undertaken a single filter monitoring withthe same cadence, then we can achieve the TDC1 results of 400 accurate measurements and 0.25% precision in time delay distance, and hence (roughly) $H_0$ - but if we cannot, then time delay cosmography will be significantly degraded and we would need additional monitoring data.

In [ ]:
assert False

## Look at summary values across multiple runs 



In [ ]:
# fetch MAF's archive of pre-computed results: families of related OpSim runs (e.g. a 'rolling cadence'
# family), the full table of summary-metric values for every archived run, and the standard named sets
# of metrics (which maps short labels like 'SL TDC' to the actual TdcMetric summary-metric names)
families = maf.archive.get_family_descriptions()
family_list = families.index.values
summaries = maf.get_metric_summaries()
metric_set = maf.get_metric_sets()

In [ ]:
fams = [
    f for f in families.index if not f.startswith("ddf")
]  # exclude Deep Drilling Field families; keep the wide-fast-deep strategy families
these_runs = families.explode(["run"]).loc[fams][
    "run"
]  # flat list of all individual OpSim run names in those families
baseline_run = "baseline_v2.0_10yrs"  # reference run all other runs are compared/normalized to
lines = maf.find_family_lines(
    families, fams
)  # x-axis positions separating each family, for drawing vertical divider lines

kk = [
    "SL TDC"
]  # the metric-set label for this notebook's Strong-Lensing Time Delay Challenge summary metrics
for k in kk:
    # Plot two versions of the figures
    # plot_run_metric_mesh: a 2D heatmap (runs x metrics) of each SL-TDC summary value relative to the baseline run
    fig, ax = maf.plot_run_metric_mesh(
        summaries.loc[these_runs, metric_set.loc[k]["metric"]],
        baseline_run=baseline_run,
        metric_label_map=metric_set.loc[k]["short_name"],
        metric_set=metric_set.loc[k],
        color_range=0.3,
    )
    fig.set_figwidth(15)
    for l in lines:
        ax.axvline(l, color="k", alpha=0.5)

for k in kk:
    # plot_run_metric: a per-run scatter/line plot of each SL-TDC summary metric (normalized to the baseline),
    # letting you see the value for every individual OpSim run/strategy side by side
    fig, ax = maf.plot_run_metric(
        summaries.loc[these_runs, metric_set.loc[k]["metric"]],
        baseline_run=baseline_run,
        metric_set=metric_set.loc[k],
        metric_label_map=metric_set.loc[k]["short_name"],
        horizontal_quantity="run",
        vertical_quantity="value",
    )
    fig.set_figwidth(15)
    ylims = list(ax.get_ylim())
    if ylims[0] < 0.5:
        ylims[0] = 0.5
    if ylims[1] > 2:
        ylims[1] = 2
    ax.set_ylim(ylims)
    lgd = plt.legend(loc=(1.01, 0.0), fancybox=True, numpoints=1, fontsize="medium")
    for l in lines:
        ax.axvline(l - 0.05, color="k", alpha=0.5)

We see the greatest variation in the rolling cadence runs. The rolling cadence is potentially a weak spot for this metric; the rolling cadence distributes visits unevenly over time, which may not be captured appropriately by the TdcMetric (which uses a median of the season lengths and a mean of all season's cadence rates). The metric does add limiting magnitude cuts, so short exposures should be discounted. The suppress repeats series shows improvement, as the cadence should increase as visits are moved to different nights; a related series which moves visits among nights if the triplets family, which shows mixed effects. 